In [69]:
import config as cfg
import random

In [71]:
def _resolve(enemy_action, player_action):
    dmg_to_player = 0
    dmg_to_enemy = 0

    # --- Scenario 1: Both Attack ---
    if enemy_action == cfg.ACTION_ATTACK and player_action == cfg.ACTION_ATTACK:
        
        dmg_to_player += cfg.ATTACK_DAMAGE
        dmg_to_enemy += cfg.ATTACK_DAMAGE

    # --- Scenario 2: Enemy Attacks, Player Defends ---
    elif enemy_action == cfg.ACTION_ATTACK and player_action == cfg.ACTION_DEFEND:

        dmg_to_player += cfg.ATTACK_DAMAGE * cfg.DEFEND_CHIP_DAMAGE_RATIO

    # --- Scenario 3: Player Attacks, Enemy Defends ---
    elif player_action == cfg.ACTION_ATTACK and enemy_action == cfg.ACTION_DEFEND:

        dmg_to_enemy += cfg.ATTACK_DAMAGE * cfg.DEFEND_CHIP_DAMAGE_RATIO

    # --- Scenario 4: Enemy Attacks, Player Dodges ---
    elif enemy_action == cfg.ACTION_ATTACK and player_action == cfg.ACTION_DODGE_COUNTER:

        if random.random() < cfg.DODGE_COUNTER_SUCCESS_RATE:
            dmg_to_enemy += cfg.DODGE_COUNTER_DAMAGE
        else:
            dmg_to_player += cfg.DODGE_COUNTER_FAIL_DAMAGE

    # --- Scenario 5: Player Attacks, Enemy Dodges ---
    elif player_action == cfg.ACTION_ATTACK and enemy_action == cfg.ACTION_DODGE_COUNTER:

        if random.random() < cfg.DODGE_COUNTER_SUCCESS_RATE:
            dmg_to_player += cfg.DODGE_COUNTER_DAMAGE
        else:
            dmg_to_enemy += cfg.DODGE_COUNTER_FAIL_DAMAGE

    return dmg_to_player, dmg_to_enemy

In [72]:
states = ["Attack", "Defend", "Dodge"]

In [73]:
def _handle_player_choices(hp):
    hp_percentage = hp / cfg.MAX_HP
        
    if(hp_percentage > cfg.PHASE_HIGH_HP):
        weights = list(cfg.PHASE_WEIGHTS["aggressive"])
        state = "Agressive"
    elif(hp_percentage < cfg.PHASE_LOW_HP):
        weights = list(cfg.PHASE_WEIGHTS["defensive"])
        state = "Defensive"
    else:
        weights = list(cfg.PHASE_WEIGHTS["balanced"])
        state = "Balanced"
        
    choice = random.choices([cfg.ACTION_ATTACK, cfg.ACTION_DEFEND, cfg.ACTION_DODGE_COUNTER], weights = weights, k = 1)[0]
    return choice, state

In [ ]:
def _handle_agent_choices(hp):
    hp_percentage = hp / cfg.MAX_HP
    
    if(hp_percentage > cfg.PHASE_HIGH_HP):
        weights = list(cfg.PHASE_WEIGHTS["aggressive"])
        state = "Agressive"
    elif(hp_percentage < cfg.PHASE_LOW_HP):
        weights = list(cfg.PHASE_WEIGHTS["defensive"])
        state = "Defensive"
    else:
        weights = list(cfg.PHASE_WEIGHTS["balanced"])
        state = "Balanced"

    weights = [0.70  , 0.05  , 0.25] #OVERRIDE
    state = "Agressive" #OVERRIDE

    choice = random.choices([cfg.ACTION_ATTACK, cfg.ACTION_DEFEND, cfg.ACTION_DODGE_COUNTER], weights = weights, k = 1)[0]
    return choice, state

In [95]:
player_hp, agent_hp = 100, 100
while(player_hp and agent_hp > 0):

    player_action, player_state = _handle_player_choices(player_hp)
    agent_action, agent_state = _handle_agent_choices(agent_hp)

    player_dmg, agent_dmg = _resolve(agent_action, player_action)
    player_hp -= player_dmg
    agent_hp -= agent_dmg

    player_hp = max(0, player_hp)
    agent_hp = max(0, agent_hp)
    
    print(
    f"--- Turn Summary ---\n"
    f"Actions : Player -> {states[player_action]:<8}  |  Agent -> {states[agent_action]}\n"
    f"Taken   : Player -> {player_dmg:<8.1f}  |  Agent -> {agent_dmg:.1f}\n"
    f"HP      : Player -> {player_hp:<8.1f}  |  Agent -> {agent_hp:.1f}\n"
    f"State   : Player -> {player_state} | Agent -> {agent_state:}\n"
    )

if(player_hp == 0 and agent_hp != 0):
    print("Agent Wins!")
elif(player_hp != 0 and agent_hp == 0):
    print("Player Wins!")
else:
    print("Both idiots killed eachother")

--- Turn Summary ---
Actions : Player -> Attack    |  Agent -> Attack
Taken   : Player -> 10.0      |  Agent -> 10.0
HP      : Player -> 90.0      |  Agent -> 90.0
State   : Player -> Agressive | Agent -> Agressive

--- Turn Summary ---
Actions : Player -> Attack    |  Agent -> Attack
Taken   : Player -> 10.0      |  Agent -> 10.0
HP      : Player -> 80.0      |  Agent -> 80.0
State   : Player -> Agressive | Agent -> Agressive

--- Turn Summary ---
Actions : Player -> Dodge     |  Agent -> Attack
Taken   : Player -> 15.0      |  Agent -> 0.0
HP      : Player -> 65.0      |  Agent -> 80.0
State   : Player -> Agressive | Agent -> Agressive

--- Turn Summary ---
Actions : Player -> Dodge     |  Agent -> Attack
Taken   : Player -> 0.0       |  Agent -> 12.0
HP      : Player -> 65.0      |  Agent -> 68.0
State   : Player -> Agressive | Agent -> Agressive

--- Turn Summary ---
Actions : Player -> Dodge     |  Agent -> Attack
Taken   : Player -> 0.0       |  Agent -> 12.0
HP      : Player -> 